# EDS conversion notebook
This notebook reads EDS data from FEMTUS (works with FEMTUS version 1.x.x and verified with JIF format version 1069064) into hyperspy. The notebook requires a python environment with 

- `hyperspy`
- [`exspy`](https://exspy.readthedocs.io/en/latest/user_guide/index.html#user-guide)
- `h5py`
- `numpy`
- `json`
- `zarr`

The EDS data from FEMTUS is saved as a .jh5 file that contains the _summed_ spectrum at each scan position (i.e. an image showing the total Xray intensity at each probe position), and the raw data is saved as separated .hdf5 files in the subdirectory "<jh5_file_name>/WL/Spectralimage/". This notebook loads the .jh5 file (actually just a .hdf5 file) that contains the metadata and then attempts to load any data located in the subdirectory.

Note that you can always use [HDFView](https://www.hdfgroup.org/download-hdfview/) software to see the structure of a .hdf5 file (this includes .jh5 and .hspy files).

In [ ]:
%matplotlib qt
import hyperspy.api as hs
import exspy as exs
import numpy as np
from pathlib import Path
import h5py
import json
import logging
from typing import Union, Dict, Any, Optional, Tuple, AnyStr, List
from zarr import ZipStore

In [54]:
def _convert_bytes_to_str(value: Any) -> AnyStr:
    """
    Recursively convert bytes objects to strings, handling nested data structures.

    This function processes various data types and converts any bytes or numpy.bytes_
    objects to UTF-8 decoded strings. It handles nested structures like lists,
    tuples, NumPy arrays, and dictionaries by recursively applying the conversion
    to their elements or values.

    Parameters
    ----------
    value : Any
        The value to process, which may be bytes, numpy.bytes_, a list/tuple/ndarray
        containing bytes, a dictionary with bytes values, or other types.

    Returns
    -------
    Any
        The processed value with all bytes objects converted to strings where applicable.
        For complex structures, returns a new structure with converted elements.

    Examples
    --------
    >>> _convert_bytes_to_str(b'hello')
    'hello'

    >>> _convert_bytes_to_str([b'item1', b'item2'])
    ['item1', 'item2']

    >>> _convert_bytes_to_str({'key': b'value'})
    {'key': 'value'}
    """
    # Check if the current value is a bytes object
    if isinstance(value, bytes):
        # Decode bytes to a UTF-8 string
        return value.decode("utf-8")

    # Check if the current value is a numpy.bytes_ object (common in HDF5 files)
    elif isinstance(value, np.bytes_):
        return value.decode("utf-8")

    # Check if the value is a list, tuple, or NumPy array
    elif isinstance(value, (list, tuple, np.ndarray)):
        if len(value) == 1:
            return _convert_bytes_to_str(value[0])
        else:
            # Otherwise, recursively process each element in the container
            return [_convert_bytes_to_str(item) for item in value]
    elif isinstance(value, dict):
        return {k: _convert_sbytes_to_str(v) for k, v in value.items()}

    # If none of the above conditions match, return the value unchanged
    else:
        return value

def load_EDSjh5(path: Path):
    """
    Load a jh5 EDS file.

    If there is a spectrum image present under `/<filename.stem>/WL/Spectralimaging`, loads this data instead and adds the .jh5 file in the given path in the metadata instead.

    Parameters
    ----------
    path : str or Path
        The path to a .jh5 EDS file. 

    Returns
    -------
    signal : hs.api.signals.Signal2D
        The EDS signal. HDF5 attributes from the file is added to the optional metadata.
    """
    path = Path(path)
    if not path.suffix == '.jh5':
        raise ValueError(f'Cannot load file "{path.absolute()}". Can only load .jh5 files.')
    with h5py.File(path, "r") as jh5_file:
        dataset = jh5_file["0"]
        data = np.squeeze(np.array(jh5_file["0"]))
        metadata_dict = get_attr(dataset)
    signal = hs.signals.Signal2D(data)
    signal.original_metadata.add_dictionary({'JH5': metadata_dict})
    datatype = metadata_dict.get('DataType', None)
    if datatype is not None:
        if datatype == 'EdsCube':
            signal.set_signal_type('EDS_TEM')
    signal.metadata.General.title = path.stem
    
    #Check for EDS spectrum data location
    try:
        expected_path = path.parent / path.stem / "WL" / "Spectralimaging"
        spectrum_image, metadata_dict = load_spectrum_image(expected_path)
    except Exception as e:
        logging.debug(f"No Spectrum image found due to error {e}", exc_info=True)
    else:
        spectrum_signal = hs.signals.Signal2D(spectrum_image).T
        spectrum_signal.set_signal_type('EDS_TEM')
        metadata_dict["Summed"] = signal
        spectrum_signal.original_metadata.add_dictionary({"JH5": metadata_dict})
        spectrum_signal.metadata.General.title = path.stem
        set_metadata_from_JH5metadata(spectrum_signal, signal.original_metadata.as_dictionary())
        set_calibration_from_JH5metadata(spectrum_signal, signal.original_metadata.as_dictionary())
        signal = spectrum_signal
    
    return signal

def set_metadata_from_JH5metadata(signal: exs.signals.EDSTEMSpectrum, metadata: Dict) -> Tuple[float, float, float]:
    """
    Sets metadata and calibrations based on a JH5 metadata dictionary
    """
    ht_keys = ["JH5", "OptionalData", "Information", "Tags", "HT", "AccelerationVoltage"]
    live_time_keys = ["JH5", "OptionalData", "Information", "Tags", "EDS", "DetectorInformation", "FirstDetector", "LiveTime"]
    real_time_keys = ["JH5", "OptionalData", "Information", "Tags", "EDS", "DetectorInformation", "FirstDetector", "RealTime"]
    


    ht = get_value_from_dict(metadata, ht_keys)
    live_time = get_value_from_dict(metadata, live_time_keys)
    real_time = get_value_from_dict(metadata, real_time_keys)

    signal.set_microscope_parameters(beam_energy=ht*1E-3, live_time=live_time, real_time=real_time)

    return (ht, live_time, real_time)

def set_calibration_from_JH5metadata(signal: exs.signals.EDSTEMSpectrum, metadata: Dict) -> List:
    """
    Sets the calibration of the signal based on a JH5 metadata dictionary
    """
    calibration_keys = ["JH5", "OptionalData", "Information", "MeasurementInformation", "CalibrationCoefficients"]
    calibration = get_value_from_dict(metadata, calibration_keys)

    for ax, cal in enumerate(calibration):
        unit = cal["Unit"]
        unit = unit[:1].lower() + unit[1:]
        signal.axes_manager[ax].scale = cal["Scale"]
        signal.axes_manager[ax].units = unit
        signal.axes_manager[ax].offset = cal["Offset"]
        signal.axes_manager[ax].convert_to_units()

    return calibration

def get_value_from_dict(dictionary: Dict, key_list: List) -> Any:
    """
    Return a value from a nested dictionary given an sequenced list of keys
    """
    value = dictionary
    while len(key_list)>0:
        key = key_list.pop(0)
        value = value.get(key, {})
    return value
        

def load_spectrum_image(path: Path) -> Tuple[np.ndarray, Dict]:
    """
    Load the spectrum image parts and concatenate them
    """
    parts = []
    if path.exists():
        file_list = [f"{str(file.stem)}" for file in path.iterdir() if file.is_file()]
        logging.debug(f"Files in spectrum imaging directory:\n\t{'\n\t'.join(file_list)}")
        metadata_dict = {}
        for f in path.glob('Part-*.hdf5'):
            with h5py.File(f, "r") as part_file:
                print(f'Loaded file "{f}"')
                dataset = part_file["DefaultDataset"]
                parts.append(np.squeeze(np.array(dataset)))
                metadata_dict[f.stem] = get_attr(dataset)
        data = np.concatenate(parts, axis=0).T
        return data, metadata_dict
    else:
        raise FileNotFoundError(f'Cannot find expected Spectral imaging directory "{path}"')

def get_attr(dataset: h5py.Dataset) -> Dict:
    """
    Return the attributes of a dataset as a dictionary
    """
    # Retrieve attributes of the dataset
    attrs = dict(dataset.attrs) if len(dataset.attrs) > 0 else {}
    # Convert attributes, handling potential bytes or nested strings
    return {key: str2dict(_convert_bytes_to_str(value)) for key, value in attrs.items()}

def str2dict(string: Union[str, None]) -> Dict[str, Any]:
    """
    Convert a JSON-formatted string representation of a dictionary into an actual Python dictionary.

    This function attempts to parse a string that represents a dictionary in JSON format.
    If the string contains nested dictionaries represented as strings, those will also be converted.

    Parameters
    ----------
    string : str or None
        A string representation of a dictionary in JSON format. Can contain nested
        dictionaries as strings that will be recursively converted.

    Returns
    -------
    dict
        A Python dictionary parsed from the input string. If the input is None or an
        empty string, returns an empty dictionary.

    Raises
    ------
    TypeError
        If the string cannot be parsed as JSON or if it's not a valid dictionary format.

    Examples
    --------
    >>> str2dict('{"key": "value"}')
    {'key': 'value'}

    >>> str2dict('{"nested": "{\\"inner\\": \\"value\\"}"}')
    {'nested': {'inner': 'value'}}
    """
    # Check if the input is a string
    if isinstance(string, str):
        # Replace single quotes with double quotes to make it valid JSON
        string = string.replace("'", '"')

        # Check if the string is non-empty
        if len(string) > 0:
            try:
                # Attempt to parse the string as JSON into a dictionary
                dictionary = json.loads(string)
            except json.JSONDecodeError as e:
                raise TypeError(
                    f'Could not convert string "{string}" to a dictionary using json: {str(e)}'
                )
            except Exception as e:
                # Catch any other unexpected errors and re-raise them as TypeError
                raise TypeError(
                    f'Unexpected error converting string "{string}" to a dictionary: {str(e)}'
                )
            else:
                # If parsing was successful and result is a dictionary
                if isinstance(dictionary, dict):
                    # Create a copy of the dictionary to avoid modifying during iteration
                    dict_copy = dictionary.copy()
                    for key, value in dict_copy.items():
                        if isinstance(value, str):
                            try:
                                dictionary[key] = str2dict(value)
                            except Exception as e:
                                # If recursive conversion fails, keep the original value
                                pass
                # Return the final processed dictionary
                return dictionary
        else:
            return {}
            # raise TypeError(f'Cannot convert empty string to dictionary')
    else:
        # If input is not a string, return an empty dictionary
        return {}

## Loading
Load the EDS area cube and verify metadata and calibrations

In [55]:
path = Path(r"C:\Users\emilc\OneDrive - NTNU\NORTEM\Data\GrandARM\EDS_Magnus\20260513_Magnus\20260513_Magnus\0513_0907\EDS Area Cube_0513_090754.jh5")
signal = load_EDSjh5(path)
signal

Loaded file "C:\Users\emilc\OneDrive - NTNU\NORTEM\Data\GrandARM\EDS_Magnus\20260513_Magnus\20260513_Magnus\0513_0907\EDS Area Cube_0513_090754\WL\Spectralimaging\Part-0.hdf5"
Loaded file "C:\Users\emilc\OneDrive - NTNU\NORTEM\Data\GrandARM\EDS_Magnus\20260513_Magnus\20260513_Magnus\0513_0907\EDS Area Cube_0513_090754\WL\Spectralimaging\Part-1.hdf5"


<EDSTEMSpectrum, title: EDS Area Cube_0513_090754, dimensions: (291, 255|4096)>

Plot the data

In [ ]:
s.plot()

Print the axes manager to see the calibrations

In [56]:
signal.axes_manager

Navigation axis name,size,index,offset,scale,units
,291,0,0.0,0.01953125,µm
,255,0,0.0,0.01953125,µm
Signal axis name,size,,offset,scale,units
,4096,,0.0,0.001,MeV


Print the signal metadata

In [46]:
signal.metadata

├── Acquisition_instrument
│   └── TEM
│       ├── Detector
│       │   └── EDS
│       │       ├── azimuth_angle = 0.0
│       │       ├── elevation_angle = 35.0
│       │       ├── energy_resolution_MnKa = 130.0
│       │       ├── live_time = 67.45
│       │       └── real_time = 74.64
│       ├── Stage
│       │   └── tilt_alpha = 0.0
│       └── beam_energy  = 200.0
├── General
│   └── title = EDS Area Cube_0513_090754
└── Signal
    └── signal_type = EDS_TEM

Print the original metadata (the raw metadata stored from FEMTUS)

In [47]:
signal.original_metadata

└── JH5
    ├── Part-0
    │   ├── JifFormatId = 100
    │   ├── JifVersion = 1
    │   ├── Metadata
    │   └── OptionalData
    ├── Part-1
    │   ├── JifFormatId = 100
    │   ├── JifVersion = 1
    │   ├── Metadata
    │   └── OptionalData
    └── Summed = <Signal2D, title: EDS Area Cube_0513_090754, dimensions: (|255, 291)>

Print the original metadata of the sum spectrum .jh5 file. This should actually correspond to the metadata for the EDS area cube

In [48]:
signal.original_metadata.JH5.Summed.original_metadata

└── JH5
    ├── JifFormatId = 1069064
    ├── JifVersion = 2
    ├── Metadata
    │   ├── Accelerating Voltage
    │   ├── AcqusitionTime = 05/13/2026 09:07:54
    │   ├── CameraLength
    │   ├── DataBits
    │   ├── DataDetectorTypeID = 2
    │   ├── DataID = 17cbde2b-1d5e-40bd-80fc-50960d4d98eb
    │   ├── DataType = EdsCube
    │   ├── DimensionUnits
    │   ├── Magnification
    │   ├── MapEndEnergy
    │   ├── MapStartEnergy
    │   ├── MaxData = inf
    │   ├── MinData = -inf
    │   ├── PixelsPerUnit
    │   ├── RockingAngle
    │   ├── StartEnergy
    │   └── Version = 1
    └── OptionalData
        ├── Information
        │   ├── DataInformation
        │   │   ├── Channel = 1
        │   │   ├── ChannelType = GrayScale
        │   │   ├── DataBytes = 4
        │   │   ├── DimensionLength = 2
        │   │   ├── Dimensions = [255, 291]
        │   │   └── TypeInfo = 2
        │   ├── Header
        │   │   ├── ClumpId = 8a05c577-193a-4312-93e6-e4f8ebd51835
        │   │   ├── ClumpType = EdsCube
        │   │   ├── DataSubType = 
        │   │   ├── DataType = EDS_Cube
        │   │   ├── DetectorType = 0
        │   │   ├── Name = EDS Area Cube_0513_090754
        │   │   └── Version = 1.0.0
        │   ├── MeasurementInformation
        │   │   └── CalibrationCoefficients <list>
        │   │       ╠══ [0]
        │   │       ║   ╠══ Offset = 0.0
        │   │       ║   ╠══ Scale = 19.53125
        │   │       ║   ╚══ Unit = Nanometer
        │   │       ╠══ [1]
        │   │       ║   ╠══ Offset = 0.0
        │   │       ║   ╠══ Scale = 19.53125
        │   │       ║   ╚══ Unit = Nanometer
        │   │       ╚══ [2]
        │   │           ╠══ Offset = 0.0
        │   │           ╠══ Scale = 1
        │   │           ╚══ Unit = KeV
        │   └── Tags
        │       ├── Aperture
        │       │   └── ApertureHoleString = 1
        │       ├── DataAdditionalInformation
        │       │   ├── FFT = False
        │       │   └── ReciprocalSpace = False
        │       ├── Detector
        │       │   ├── BinningSize
        │       │   │   ├── X = 1
        │       │   │   └── Y = 1
        │       │   ├── DetectorKind = DFI_U
        │       │   ├── ExposureTimeIndex = 
        │       │   ├── ExposureTimeValue = 1.0
        │       │   ├── FrameIntegration = 1
        │       │   ├── GainIndex = 
        │       │   ├── ImagingArea
        │       │   │   ├── Height = 256
        │       │   │   ├── Width = 2048
        │       │   │   ├── X = 0
        │       │   │   └── Y = 0
        │       │   ├── Manufacturer = JEOL Ltd.
        │       │   ├── ModelCode = 
        │       │   ├── ModelDisplayName = 
        │       │   ├── OffsetIndex = 
        │       │   └── PixelsPerMeter
        │       │       ├── Horizontal = 2560.0
        │       │       └── Vertical = 2560.0
        │       ├── EDS
        │       │   ├── DetectorInformation
        │       │   │   └── FirstDetector
        │       │   │       ├── DetectorName = First
        │       │   │       ├── DetectorType = EX-34400MNU
        │       │   │       ├── InputCountrate = 9533
        │       │   │       ├── LiveTime = 67.45
        │       │   │       ├── OutputCountrate = 7868
        │       │   │       ├── PortType = ED2
        │       │   │       ├── ProcessTimeTemplateName = T4
        │       │   │       └── RealTime = 74.64
        │       │   └── SpectrumEnergyChannel = 2048
        │       ├── EOS
        │       │   ├── CalibratedCameraLength = 150.0
        │       │   ├── CalibratedMagnificationValue = 20000.0
        │       │   ├── CameraLength = 150.0
        │       │   ├── CameraLengthCalibrated = False
        │       │   ├── CameraLengthString = 15cm
        │       │   ├── ConvergenceAngleAlphaNumber = 2
        │       │   ├── ConvergenceAngleAlphaString = 2
        │       │   ├── ConvergenceAngleAlphaUnitString = 2
        │       │   ├── ConvergenceAngleAlphaValue = 0.0
        │       │   ├── DefocusString = 0.1nm

## Save the signal to zspy
Note: Don't mind the many UserWarnings that will appear

In [49]:
store = ZipStore(path.with_suffix('.zspy'), mode='w')
signal.save(store)

c:\Users\emilc\AppData\Local\miniforge3\envs\pyxem0.21.0\Lib\zipfile\__init__.py:1642: UserWarning: Duplicate name: '.zattrs'
  return self._open_to_write(zinfo, force_zip64=force_zip64)
c:\Users\emilc\AppData\Local\miniforge3\envs\pyxem0.21.0\Lib\zipfile\__init__.py:1642: UserWarning: Duplicate name: 'Experiments/EDS Area Cube_0513_090754/axis-0/.zattrs'
  return self._open_to_write(zinfo, force_zip64=force_zip64)
c:\Users\emilc\AppData\Local\miniforge3\envs\pyxem0.21.0\Lib\zipfile\__init__.py:1642: UserWarning: Duplicate name: 'Experiments/EDS Area Cube_0513_090754/axis-1/.zattrs'
  return self._open_to_write(zinfo, force_zip64=force_zip64)
c:\Users\emilc\AppData\Local\miniforge3\envs\pyxem0.21.0\Lib\zipfile\__init__.py:1642: UserWarning: Duplicate name: 'Experiments/EDS Area Cube_0513_090754/axis-2/.zattrs'
  return self._open_to_write(zinfo, force_zip64=force_zip64)
c:\Users\emilc\AppData\Local\miniforge3\envs\pyxem0.21.0\Lib\zipfile\__init__.py:1642: UserWarning: Duplicate name: '

## Load the saved zspy signal again to verify

In [50]:
store = ZipStore(path.with_suffix('.zspy'), mode='r')
s = hs.load(store, lazy=True)
s.metadata

├── Acquisition_instrument
│   └── TEM
│       ├── Detector
│       │   └── EDS
│       │       ├── azimuth_angle = 0.0
│       │       ├── elevation_angle = 35.0
│       │       ├── energy_resolution_MnKa = 130.0
│       │       ├── live_time = 67.45
│       │       └── real_time = 74.64
│       ├── Stage
│       │   └── tilt_alpha = 0.0
│       └── beam_energy  = 200.0
├── General
│   ├── FileIO
│   │   ├── 0
│   │   │   ├── hyperspy_version = 2.3.0
│   │   │   ├── io_plugin = rsciio.zspy
│   │   │   ├── operation = save
│   │   │   └── timestamp = 2026-06-16T09:59:18.095757+02:00
│   │   └── 1
│   │       ├── hyperspy_version = 2.3.0
│   │       ├── io_plugin = rsciio.zspy
│   │       ├── operation = load
│   │       └── timestamp = 2026-06-16T09:59:18.850287+02:00
│   └── title = EDS Area Cube_0513_090754
└── Signal
    └── signal_type = EDS_TEM

In [51]:
s.axes_manager

Navigation axis name,size,index,offset,scale,units
,291,0,0.0,1.0,
,255,0,0.0,1.0,
Signal axis name,size,,offset,scale,units
,4096,,0.0,1.0,


In [52]:
s.plot()

  0%|          | 0/33 [00:00<?, ?it/s]

## Interactive operations
### Subarea analysis of the EDS area cube

In [ ]:
analysis = 'mean' #Choose between 'sum', 'mean', 'std', 'max', 'min'
s.plot()
roi = hs.roi.RectangularROI()
signal_roi = roi.interactive(s, color='red')
analysis_options = {'sum': signal_roi.sum, 'mean': signal_roi.mean, 'std': signal_roi.std, 'max': signal_roi.max, 'min': signal_roi.min}
roi_spectrum = hs.interactive(analysis_options[analysis], event=roi.events.changed, recompute_out_event=None)
roi_spectrum.plot()

  0%|          | 0/23 [00:00<?, ?it/s]

### Energy windows

In [32]:
hs.plot.plot_roi_map(s, rois=2, navigator=s.sum())

([SpanROI(left=0, right=1.02375), SpanROI(left=1.02375, right=2.0475)],
 [<LazySignal2D, title: Integrated intensity, dimensions: (|291, 255)>,
  <LazySignal2D, title: Integrated intensity, dimensions: (|291, 255)>])